<a href="https://colab.research.google.com/github/vgasztany/thesis/blob/main/szakdolgozat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import train_test_split
from sklearn import svm

In [ ]:
!wget 'https://drive.google.com/uc?export=download&id=1Q2e5630PNqasosgf1XV1olflYjtFaIkg' -O './1951_2020_allgauges.csv'
df = pd.read_csv('1951_2020_allgauges.csv', index_col=[0])

In [ ]:
szeged = df['2275']
szeged.head()

In [ ]:
check_nan = szeged.isnull().values.any()
print(check_nan)

In [ ]:
ax = szeged.plot(figsize=(30,15))
ax.set_xlabel("Date")
ax.set_ylabel("Value")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf

plot_acf(szeged, lags=100)
plot_pacf(szeged, lags=20)

In [ ]:
#ADF Statistic>Critical Values->non-stationary
#ADF Statistic<Critical Values->stationary

result = adfuller(szeged, autolag='AIC')
print(f'ADF Statistic: {result[0]}')
print(f'p-value: {result[1]}')
for key, value in result[4].items():
    print('Critial Values:')
    print(f'   {key}, {value}')

In [ ]:
# Split the data into train and test sets

train_size = int(len(szeged)*0.7)
train_data, test_data = szeged.iloc[:train_size], szeged.iloc[train_size:]
print(train_data)
print(test_data)

In [ ]:
#Standardize Data
# CSak a train mean,std kell, azt használjuk a test set-re is
normalized_train_data = pd.DataFrame()
normalized_train_data['Results']= (train_data - train_data.mean()) / train_data.std()
normalized_train_data

In [ ]:
#Standardize Data
# CSak a train mean,std kell, azt használjuk a test set-re is
normalized_test_data = pd.DataFrame()
normalized_test_data['Results']= (test_data - train_data.mean()) / train_data.std()
normalized_test_data

In [ ]:
#Create the matrix of train datas
def create_train_matrix(order):
  matrix = []
  for x in range(len(normalized_train_data) - order):
    line = normalized_train_data.iloc[x:(x+order+1)]
    line = line.values.tolist()
    matrix.append(line)
  train_matrix = pd.DataFrame(matrix)
  return train_matrix

In [ ]:
#Create the matrix of test datas
def create_test_matrix(order):
  matrix = []
  for x in range(len(normalized_test_data) - order):
    line = normalized_test_data.iloc[x:(x+order+1)]
    line = line.values.tolist()
    matrix.append(line)

  test_matrix = pd.DataFrame(matrix)
  return test_matrix

In [ ]:
from pandas.core.frame import validate_axis_style_args
# grid search ARIMA parameters for time series
import warnings
from math import sqrt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# evaluate an ARIMA model for a given order (p,d,q)
def evaluate_arima_model(X, arima_order):
	# prepare training dataset
	train_size = int(len(X) * 0.66)
	train, val = X[0:train_size], X[train_size:]
	history = [x for x in train]
	# make predictions
	predictions = list()
	for t in range(len(val)):
		model = ARIMA(history, order=arima_order)
		model_fit = model.fit()
		yhat = model_fit.forecast()[0]
		predictions.append(yhat)
		history.append(val[t])
	# calculate out of sample error
	rmse = sqrt(mean_squared_error(validate_axis_style_args, predictions))
	return rmse

# evaluate combinations of p, d and q values for an ARIMA model
def evaluate_models(dataset, p_values, d_values, q_values):
	dataset = dataset.astype('float32')
	best_score, best_cfg = float("inf"), None
	for p in p_values:
		for d in d_values:
			for q in q_values:
				order = (p,d,q)
				try:
					rmse = evaluate_arima_model(dataset, order)
					if rmse < best_score:
						best_score, best_cfg = rmse, order
					print('ARIMA%s RMSE=%.3f' % (order,rmse))
				except:
					continue
	print('Best ARIMA%s RMSE=%.3f' % (best_cfg, best_score))

# evaluate parameters
p_values = [1]
d_values = range(0, 3)
q_values = range(0, 20)
warnings.filterwarnings("ignore")
evaluate_models(normalized_train_data.values, p_values, d_values, q_values)

In [ ]:
#Split matrix into k-fold
from statsmodels.tsa.arima.model import ARIMA
from pandas import DataFrame

model = ARIMA(normalized_test_data, order=(60,1,0))
model_fit = model.fit()
print(model_fit.summary())
residuals = DataFrame(model_fit.resid)
residuals.plot()
plt.show()
residuals.plot(kind='kde')
plt.show()
print(residuals.describe())